# RandomUnderSampler + Random Forest
**tabular › imbalanced** · notebook `undersample_rf`

**Kapan dipakai:** Data sangat besar sehingga membuang sebagian mayoritas mempercepat latihan.

**Jebakan umum:** Membuang informasi mayoritas. Jangan seimbangkan 1:1 penuh bila rasio asli sangat timpang.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular_extra/imbalanced/mammography_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular_extra/imbalanced/mammography_test.csv"      # None kalau tidak ada file test
TARGET     = "label"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "average_precision"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "undersample_rf"
TASK       = "classification"
KIND       = "tree"
OUT = out_dir("tabular", "imbalanced", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
minority = df[TARGET].value_counts().idxmin()          # kelas langka dijadikan kelas 1 (positif)
y = (df[TARGET] == minority).astype(int).to_numpy()
print(f"positif (kelas 1) = {minority!r}: {y.sum()} dari {len(y)} ({y.mean():.2%})")
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Kenapa accuracy menipu di sini
Model yang selalu menebak kelas mayoritas sudah punya accuracy tinggi tetapi tidak berguna. Pakai **PR-AUC**, F1 kelas positif, recall/precision.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score
dm = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
print(f"selalu tebak mayoritas: accuracy={accuracy_score(yva, dm.predict(Xva)):.4f}, F1 positif={f1_score(yva, dm.predict(Xva), zero_division=0):.4f}")

## 5. Model + penyeimbang
`sampler` (bila ada) hanya bekerja saat **fit** di dalam pipeline, jadi tidak pernah menyentuh data validation.

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from superdb.tabular import make_preprocessor

sampler = RandomUnderSampler(sampling_strategy=0.2, random_state=42)   # minoritas = 20% dari mayoritas
model = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, n_jobs=-1, random_state=42)
steps = [("pre", make_preprocessor(NUM, CAT, kind=KIND))] + ([("sampler", sampler)] if sampler is not None else []) + [("model", model)]
pipe = ImbPipeline(steps)
pipe

## 6. Cross-validation dengan PR-AUC

In [ ]:
folds, summary = cv_score(pipe, Xtr, ytr, TASK, SCORING, N_SPLITS, SEED)
display(folds.round(4)); print(summary)

## 7. Latih & evaluasi (threshold 0.5)

In [ ]:
pipe.fit(Xtr, ytr)
proba = pipe.predict_proba(Xva)[:, 1]
metrics = classification_metrics(yva, (proba >= 0.5).astype(int), proba, classes=[0, 1])
display(show_table(metrics))
from superdb.tabular_extra import best_f1_threshold, plot_pr_curve
plot_pr_curve(yva, proba, path=OUT / "pr_curve.png"); plt.show()

## 8. Atur threshold
Threshold 0.5 jarang tepat untuk data timpang. Threshold dicari dari prediksi **out-of-fold pada training** (bukan dari validation, agar tidak curang), lalu diterapkan ke validation.

In [ ]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
oof = cross_val_predict(pipe, Xtr, ytr, cv=StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
THR, f1_oof = best_f1_threshold(ytr, oof)
print(f"threshold terbaik dari OOF = {THR:.3f} (F1 OOF {f1_oof:.3f})")
pred_thr = (proba >= THR).astype(int)
from sklearn.metrics import precision_score, recall_score
rows = {f"thr=0.5": ((proba >= .5).astype(int)), f"thr={THR:.3f}": pred_thr}
display(pd.DataFrame({k: dict(precision=precision_score(yva, v, zero_division=0), recall=recall_score(yva, v), f1=f1_score(yva, v, zero_division=0)) for k, v in rows.items()}).T.round(3))
plot_confusion(yva, pred_thr, labels=[0, 1], path=OUT / "confusion.png"); plt.show()
metrics["threshold"] = THR; metrics["f1_at_threshold"] = float(f1_score(yva, pred_thr, zero_division=0))
save_json({"cv": summary, "valid": metrics}, OUT / "metrics.json")

## 9. Latih ulang pada semua data → submission

In [ ]:
from sklearn.base import clone
final = clone(pipe).fit(X, y)
if test is not None:
    Xte, _, _ = prepare_features(test, cols); Xte = Xte.reindex(columns=X.columns)
    ids = test[ID_COL] if ID_COL else np.arange(len(test))
    pr = final.predict_proba(Xte)[:, 1]
    sub = make_submission(ids, (final.predict_proba(Xte)[:, 1] >= THR).astype(int), ID_COL or "id", TARGET, OUT / "submission.csv")
    sub["proba"] = pr; sub.to_csv(OUT / "submission_with_proba.csv", index=False)
    display(sub.head(3)); print("label 1 pada submission:", int(sub[TARGET].sum()), "dari", len(sub))
save_model({"pipe": final, "threshold": THR, "cols": cols, "positive": minority}, OUT / "model.joblib")
b = load_model(OUT / "model.joblib")
assert np.allclose(b["pipe"].predict_proba(X.head(50)), final.predict_proba(X.head(50))), "model muat-ulang beda"
print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan beberapa penyeimbang di folder ini; yang terbaik bergantung data, jadi selalu uji.
- Kalau metrik lomba F1/recall, jangan lupa `THR` ikut disimpan (sudah di `model.joblib`).
- Kalibrasi probabilitas dan analisis biaya salah-prediksi: lihat `cookbook/`.